In [2]:
import numpy as np
import pandas as pd

import plotly.express as px
from pathlib import Path
import transforms as T

In [3]:
def log_bin(column, factor = 1):
    floor = np.pow(10,np.floor(np.log10(column)))
    bin = np.floor(column/floor/factor) * floor * factor
    bin = np.where(bin == 0, floor, bin).astype(int)
    return bin

In [4]:
root = Path("./data")
raw_path = root/"raw"
clean_path = root/"clean"
map_name = "data_map.csv"

data_map = pd.read_csv(root/map_name)

In [ ]:
df = pd.read_csv(raw_path/'pppub25.csv')
market_cols = [
    'MARSUPWT', # Individuals associated with income
    'WSAL_VAL', # Wages & Salaries
    'SEMP_VAL', # Self-Employment
    'FRSE_VAL', # Farm Income
    'INT_VAL',  # Interest
    'DIV_VAL',  # Dividends
    'RNT_VAL',  # Rent, Royalties, Estates, Trusts
    'RET_VAL'   # Pensions & Taxable Retirement Distributions
]

df['taxable_market_income'] = df[market_cols].sum(axis=1)


In [9]:
df = df['MARSUPWT,PTOTVAL,SS_VAL'.split(',')]
df.to_csv(raw_path/'pppub25_truncated.csv', index=False)

In [ ]:
df = pd.read_csv(raw_path/'pppub25.csv')
df.rename({"MARSUPWT":"people", "PTOTVAL":"row_usd"},axis=1, inplace=True)
df = df[df['row_usd'] > 0]
df['people']= df['people']/100
df['usd'] = df['row_usd'] * df['people']
df['bin'] = bin(df['row_usd'], factor = 1)
df[df['bin'] < 10000] = 10000
df[df['bin'] > 1000000] = 1000000
df = df.groupby('bin').agg({'people':'sum', 'usd':'sum'})
df['people_cdf'] = df['people'].cumsum()
df['usd_cdf'] = df['usd'].cumsum()


KeyError: 'people'

In [73]:
df.tail()

,people,usd,people_cdf,usd_cdf
bin,,,,
600000,256335.47,1.628815e+11,3.302905e+08,1.540940e+13
700000,148805.40,1.096837e+11,3.304393e+08,1.551909e+13
800000,80886.18,6.780990e+10,3.305202e+08,1.558690e+13
900000,76189.12,7.273128e+10,3.305964e+08,1.565963e+13
1000000,6651025.24,7.353314e+11,3.372474e+08,1.639496e+13


In [76]:
df.describe()

,people,usd,people_cdf,usd_cdf
count,1.900000e+01,1.900000e+01,1.900000e+01,1.900000e+01
mean,1.774986e+07,8.628926e+11,2.809891e+08,9.789001e+12
std,3.180501e+07,1.007669e+12,6.218628e+07,5.924235e+12
min,7.618912e+04,6.780990e+10,1.411406e+08,4.330632e+11
25%,6.172493e+05,2.880864e+11,2.456922e+08,4.558946e+12
50%,7.618431e+06,7.353314e+11,3.194680e+08,1.226344e+13
75%,2.083468e+07,9.814291e+11,3.301623e+08,1.532796e+13
max,1.411406e+08,4.642485e+12,3.372474e+08,1.639496e+13


In [82]:
px.bar(df['usd']/df['people'])

In [81]:
px.bar(df['people'])